In [44]:
import pandas as pd
 
 
url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-II/datasets/titanic.csv"
df = pd.read_csv(url)

In [45]:
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


**Pregunta A (Sumarización Categórica)**: Usen la función `.value_counts(normalize=True)` en la columna Survived (0 = Murió, 1 = Sobrevivió). ¿Cuál es la tasa de supervivencia global del barco expresada en porcentaje?


In [46]:
survived = df['Survived'].value_counts(normalize=True)

In [47]:
print(f"Porcentage of passenger that survived: {survived[1]*100}%")


Porcentage of passenger that survived: 38.38383838383838%


**Pregunta B (Agrupación y Agregación)**: El famoso código marítimo era "mujeres y niños primero". Comprobemos esto matemáticamente. Ejecuten una agrupación por la columna de género y calculen el promedio de la columna Survived. ¿Qué porcentaje exacto de mujeres sobrevivió en contraste con los hombres?

In [48]:
by_sex = df.groupby('Sex')['Survived'].mean()
print(f"Porcentage of female pasangers that survived: {by_sex['female']*100}%\nPorcentage of male passangers that survived: {by_sex['male']*100}%")

Porcentage of female pasangers that survived: 74.20382165605095%
Porcentage of male passangers that survived: 18.890814558058924%


**Pregunta C**: El director financiero nota que la tarifa máxima (Fare) cobrada fue de más de 500 libras, mientras que el promedio ronda las 32. Sospecha de outliers. Utilicen las herramientas de dispersión en Pandas para confirmarlo:
- Calculen el Cuartil 1 (25%) y el Cuartil 3 (75%) de la columna Fare usando el método .quantile([0.25, 0.75]).
- Calculen matemáticamente el Rango Intercuartílico (IQR = Q3 - Q1).
- Calculen el límite superior aceptable (Q3 + 1.5 * IQR). Escriban una línea de código para filtrar el DataFrame y descubrir exactamente cuántos pasajeros pagaron una tarifa por encima de ese límite matemático. ¿A qué clase (Pclass) pertenecían la mayoría de ellos?


In [49]:
q1 = df['Fare'].quantile(0.25)
q3 = df['Fare'].quantile(0.75)

iqr = q3 - q1

minimum = q1 - 1.5*iqr
maximum = q3 + 1.5*iqr

outliers = df[df['Fare'] > maximum] 
print(f"Number of people who paid a fare above the maximum {len(outliers)}")
print(outliers['Pclass'].value_counts())


Number of people who paid a fare above the maximum 116
Pclass
1    104
3      7
2      5
Name: count, dtype: int64



**Pregunta D**: Calculen la media y la mediana de la columna Fare. Notarán una diferencia enorme entre ambos valores. Matemáticamente, ¿qué significa que la media sea tan superior a la mediana? Si en el futuro utilizamos un algoritmo basado en distancias euclidianas (como K-Nearest Neighbors) sin escalar previamente esta variable, ¿cómo afectará esta asimetría al aprendizaje del modelo?

In [50]:
fare_median = df['Fare'].median()
fare_mean = df['Fare'].mean()

print(f"Fare median: {fare_median}\nFare mean: {fare_mean}")

Fare median: 14.4542
Fare mean: 32.204207968574636


**R** = significa que la distribución de la Fare concentra valores muy altos por el lado derecho y la mayoría de los valores recabados están debajo de la media. Si no se escala esta variable tendría practicamente el mayor peso entre todas la columnas y el modelo no tomaría tan encuenta las columnas de valores pequeños, estaría sesgado por la fare.

**Pregunta E**: En la semana 2 vimos que el muestreo aleatorio simple es peligroso. El objetivo es entrenar un modelo que prediga la supervivencia (Survived), pero las clases están desbalanceadas. Escriban el código en Pandas para extraer una muestra de exactamente 150 pasajeros garantizando que la proporción de sobrevivientes y no sobrevivientes en la muestra sea idéntica a la de la base de datos completa. ¿Qué sesgo evitan al hacer esto?


In [51]:
import sklearn
from sklearn.model_selection import train_test_split

muestra, _ = train_test_split(df,train_size=150, stratify=df['Survived'], random_state=42)
print(len(muestra))
print(muestra['Survived'].value_counts(normalize=True))

150
Survived
0    0.613333
1    0.386667
Name: proportion, dtype: float64


**R** = como ayuda que la distribución de los datos sean mas parecida a la real se evita que la muestra se vea inclinada a tomar datos de más del conjunto que representa la mayoria, le da una mejor chance a la minoria de actuar según la realidad

**Pregunta F**: Si ejecutan df.groupby('Survived')['Age'].mean(), Pandas calculará el promedio de edad de los que vivieron y los que murieron. Sin embargo, por defecto, Pandas ignora los valores NaN al calcular la media. Si resulta que la gran mayoría de las edades faltantes (NaN) pertenecían a pasajeros de 3ra clase que murieron, ¿qué sesgo estadístico estamos introduciendo involuntariamente en el resultado de esa función y cómo afectaría la inferencia de nuestro modelo?

In [52]:
df.groupby('Survived')['Age'].mean()

Survived
0    30.626179
1    28.343690
Name: Age, dtype: float64

**R** = sesgo por valores faltantes, la variable de la edad perdería su tipo neutralidad o discriminación sobre las muertes de clases bajas

**Pregunta G**: Con el cálculo del IQR en la Pregunta C, determinaron que los boletos de 512 libras son outliers matemáticos. En Machine Learning, los valores atípicos pueden representar errores de captura (ruido que aumenta el error irreducible) o casos especiales válidos (señal). Investigando la naturaleza de un barco de lujo, ¿deberíamos eliminar estas filas con .drop() antes de entrenar nuestro modelo? Justifiquen su respuesta arquitectónica.


**R** = No creo que se deberían de quitar, por la misma naturaleza del barco no es descabellado que existieran personas que en verdad pagaron por esas tarifas dentro de primera clase, si se elimina estaríamos quitanfo info de el grupo que, históricamente, fue el que más oportunidades tuvo para sobrevivir. Sí es posible hacer una relación entre los outliners con los posibles sobrevivientes que el modelo podría predecir, lo mejor sería en todo caso es hacer una injección sobre estas rows.

**Pregunta H:** Observen la columna Name. Es texto libre (dato no estructurado), pero contiene títulos ocultos como "Mr.", "Mrs.", "Miss." o "Master.". Si lograran extraer ese título usando expresiones regulares en Pandas, podrían hacer un .groupby('Titulo')['Age'].median(). ¿Por qué imputar las edades faltantes basándose en la mediana del "Título" (ej. "Master" = niño, "Mr" = adulto) sería estadísticamente superior y reduciría el error de nuestro futuro modelo, en comparación con usar la mediana global?


**R** = por que estos títulos, por lo mismo de la época, tenían definiciones muy marcados y fácil de asimilar a edades humana, haciendo que la asignación de edades sea mucho más real o cerca de la verdadera, que a si solo se le asigna a todos la mediana global.

**Pregunta I**: Calculen la varianza de la columna Survived. Dado que es una variable categórica codificada como 0 y 1, el resultado numérico estará cerca de $0.23$. Matemáticamente, ¿qué significaría si la varianza de esta variable fuera exactamente 0.0? ¿Qué pasaría si intentan entrenar un algoritmo de clasificación con un dataset donde la variable de respuesta tiene varianza 0.0?

In [53]:
var = df['Survived'].var()

print(f"Variance of the Survived column: {var}")

Variance of the Survived column: 0.2367722165474984


**R** = significa que los datos no tiene diferencias unos con los otros, en este caso sería que o todos sobrevivieron o todos murieron sin importar el peso de las demás columnas. Si se intenta hacer un modelo que caiga bajo esta cirscunstancia no serviria de nada, daría el mismo resultado siempre

**Pregunta J**: Ejecuten una agrupación por tres niveles al mismo tiempo y cuenten cuántos pasajeros hay en cada subgrupo: df.groupby(['Pclass', 'Sex', 'Embarked'])['PassengerId'].count(). Notarán que algunos subgrupos tienen 1 o 2 pasajeros. Si un algoritmo intenta extraer reglas de probabilidad de grupos tan pequeños, se enfrentará a la "Maldición de la Dimensionalidad" (Curse of Dimensionality). ¿Qué fenómeno perjudicial (sobreajuste o subajuste) ocurrirá inevitablemente si dejamos que el modelo aprenda reglas basadas en esos grupos de 1 solo pasajero?

In [54]:
df.groupby(['Pclass', 'Sex', 'Embarked'])['PassengerId'].count()

Pclass  Sex     Embarked
1       female  C            43
                Q             1
                S            48
        male    C            42
                Q             1
                S            79
2       female  C             7
                Q             2
                S            67
        male    C            10
                Q             1
                S            97
3       female  C            23
                Q            33
                S            88
        male    C            43
                Q            39
                S           265
Name: PassengerId, dtype: int64

**R** = ocurriría un sobreajuste